# 04 · Coverage & SINR without a ray tracer

One radial is a link budget. A **map** is a plan. This notebook computes whole-area
coverage, best-server association and SINR over the pilot scene in about a second, then
uses that speed to do the things a ray tracer makes you think twice about:

- sweep five bands and watch the λ² penalty eat your coverage,
- switch terrain and buildings off to *measure* what each contributes,
- drop in a third tower and see interference, not coverage, become the binding constraint.

> The maps here come from the analytical model in `ulap_demo.propagation`, not from
> Sionna. They are for intuition and sizing. The ray-traced versions in `docs/renders/`
> are the ones to quote.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
EXAMPLES = next(p for p in [here, *here.parents] if (p / "ulap_demo").is_dir())
sys.path[:0] = [str(EXAMPLES), str(EXAMPLES.parent / "ulap-scope")]

import numpy as np
import matplotlib.pyplot as plt
from ulap_demo import load_scene, PropagationModel
from ulap_demo.scene import Tower
from ulap_demo.plotting import (use_ulap_style, plot_coverage, plot_sinr,
                                plot_best_server, plot_map, overlay_scene,
                                SUNGLOW, MARBLE_WHITE)

use_ulap_style()
scene = load_scene()
print(scene.summary())
print()
model = PropagationModel(freq_hz=3.5e9, tx_power_dbm=33.0, bandwidth_hz=100e6)
print(model.describe())

## One map, three ways to read it

In [ ]:
%time cov = model.coverage(scene, cell_m=10.0)
print()
print(cov.summary(rsrp_threshold_dbm=-95, sinr_threshold_db=0))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.6))
plot_coverage(axes[0], cov, scene=scene, title="Best-server RSRP @ 3.5 GHz")
plot_best_server(axes[1], cov, scene=scene)
plot_sinr(axes[2], cov, scene=scene, title="SINR (2 towers, 2 W, 100 MHz)")
fig.tight_layout()

Three questions, three maps:

- **RSRP** — is there enough signal? (a coverage question)
- **Best server** — who serves this pixel, and where is the handover boundary?
- **SINR** — is the signal *usable*? Cell-edge pixels can have plenty of RSRP and still be
  unusable because the neighbour is just as loud.

The dark seam through the middle of the SINR map is the cell edge. That is not a coverage
hole; it is an interference problem, and it is fixed with tilt, power and reuse — never
with another site.

## Coverage as a distribution, not a percentage

"97 % coverage" hides the whole story. The CDF is what a planner argues about.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
vals = np.sort(cov.best_rsrp_dbm.ravel())
cdf = np.arange(1, vals.size + 1) / vals.size
ax.plot(vals, 100 * cdf, color=SUNGLOW, lw=2)

for thr, label in [(-85, "good"), (-95, "usable"), (-105, "edge")]:
    pct = 100 * cov.fraction_above(thr)
    ax.axvline(thr, color=MARBLE_WHITE, ls=":", lw=1, alpha=0.5)
    ax.annotate(f"{label}\n{pct:.0f}% above", (thr, 8), color=MARBLE_WHITE,
                fontsize=8.5, ha="right", xytext=(-6, 0), textcoords="offset points")

ax.set_xlabel("best-server RSRP [dBm]")
ax.set_ylabel("% of area below")
ax.set_title("Coverage CDF — the 5th percentile is the number that hurts")
ax.grid(alpha=0.25)
print(f"5th percentile RSRP: {np.percentile(cov.best_rsrp_dbm, 5):.1f} dBm")
fig.tight_layout()

## Band sweep: the λ² penalty, measured

Same towers, same power, five bands. Nothing changes but the wavelength.

In [ ]:
BANDS = [1.8, 3.5, 6.0, 10.0, 28.0]
sweep = {}
for ghz in BANDS:
    m = PropagationModel(freq_hz=ghz * 1e9, tx_power_dbm=33.0)
    sweep[ghz] = m.coverage(scene, cell_m=15.0)

print(f"{'band':>8} {'median RSRP':>13} {'5th pct':>11} {'% >= -95 dBm':>14}")
for ghz, c in sweep.items():
    med = np.median(c.best_rsrp_dbm)
    p5 = np.percentile(c.best_rsrp_dbm, 5)
    print(f"{ghz:>6} GHz {med:>10.1f} dBm {p5:>8.1f} dBm "
          f"{100 * c.fraction_above(-95):>12.1f}%")

In [ ]:
fig, axes = plt.subplots(1, len(BANDS), figsize=(22, 4.6))
top = float(np.percentile(sweep[BANDS[0]].best_rsrp_dbm, 99.5))
for ax, ghz in zip(axes, BANDS):
    c = sweep[ghz]
    ax.imshow(c.best_rsrp_dbm, origin="lower", extent=c.extent, cmap="viridis",
              vmin=top - 90, vmax=top, interpolation="nearest")
    overlay_scene(ax, scene, label_towers=False, building_alpha=0.15)
    ax.set_title(f"{ghz:g} GHz")
    ax.set_xlabel(""); ax.set_ylabel("")
fig.suptitle("Same sites, same power — only the band changes (common colour scale)", y=1.02)
fig.tight_layout()

In [ ]:
bands = np.array(BANDS)
med = np.array([np.median(sweep[g].best_rsrp_dbm) for g in BANDS])
p05 = np.array([np.percentile(sweep[g].best_rsrp_dbm, 5) for g in BANDS])

# pure free-space expectation, anchored at the lowest band: -20 log10(f/f0)
lam2 = -20 * np.log10(bands / bands[0])
ref_med, ref_p05 = med[0] + lam2, p05[0] + lam2

fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.plot(bands, med, "o-", color=SUNGLOW, lw=2, ms=8, label="median pixel")
ax.plot(bands, ref_med, "--", color=SUNGLOW, alpha=0.45, lw=1.3, label="median, λ² only")
ax.plot(bands, p05, "s-", color="#FF7A7A", lw=2, ms=7, label="5th percentile (cell edge)")
ax.plot(bands, ref_p05, "--", color="#FF7A7A", alpha=0.45, lw=1.3,
        label="5th percentile, λ² only")
ax.set_xscale("log")
ax.set_xticks(BANDS); ax.set_xticklabels([f"{b:g}" for b in BANDS])
ax.set_xlabel("frequency [GHz]"); ax.set_ylabel("best-server RSRP [dBm]")
ax.set_title("The median follows λ². The cell edge does not.")
ax.grid(alpha=0.25, which="both"); ax.legend(fontsize=8.5)

print(f"{'band':>8} {'excess loss, median':>21} {'excess loss, 5th pct':>22}")
for g, a, b_, c_, d_ in zip(BANDS, med, ref_med, p05, ref_p05):
    print(f"{g:>6} GHz {b_ - a:>17.1f} dB {d_ - c_:>18.1f} dB")
fig.tight_layout()

Look at the two curves separately — this is the whole point of mapping instead of
computing a single link budget.

The **median** pixel tracks the λ² line almost exactly, even at 28 GHz. A typical spot in
this scene has a clear path, so it only pays the wavelength penalty. The pilot's
ray-traced sweep found the same (within ~1 dB, `docs/renders/sweep_frequency.png`).

The **5th percentile** — the cell edge, the pixels that decide your site count — falls
away from λ² and keeps falling. Those pixels are shadowed, the Fresnel zone shrinks with
frequency, and obstacles that were "nearly clear" at 1.8 GHz become hard blockers. A band
plan chosen on median coverage will be wrong about exactly the users you built the
network for.

**mmWave is not an area-coverage band here** — it is fixed-wireless and hotspots.

## Ablation: what is terrain worth? What are buildings worth?

Because the model is cheap, we can turn each physical term off and measure it — an
experiment that is awkward to run in a ray tracer because it means rebuilding the scene.

In [ ]:
variants = {
    "flat, no buildings": PropagationModel(use_terrain=False, use_buildings=False),
    "buildings only":     PropagationModel(use_terrain=False, use_buildings=True),
    "terrain only":       PropagationModel(use_terrain=True,  use_buildings=False),
    "terrain + buildings": PropagationModel(use_terrain=True, use_buildings=True),
}
abl = {k: m.coverage(scene, cell_m=10.0) for k, m in variants.items()}

base = abl["flat, no buildings"]
b_med = np.median(base.best_rsrp_dbm)
b_p05 = np.percentile(base.best_rsrp_dbm, 5)
print(f"{'variant':<21} {'median':>9} {'d':>6} {'5th pct':>10} {'d':>7} {'worst pixel':>13}")
for name, c in abl.items():
    med = np.median(c.best_rsrp_dbm)
    p05 = np.percentile(c.best_rsrp_dbm, 5)
    print(f"{name:<21} {med:>6.1f} dBm {med - b_med:>+6.1f} {p05:>7.1f} dBm "
          f"{p05 - b_p05:>+6.1f} {c.best_rsrp_dbm.min():>10.1f} dBm")

In [ ]:
full = abl["terrain + buildings"]
flat = abl["flat, no buildings"]
delta = full.best_rsrp_dbm - flat.best_rsrp_dbm

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
plot_map(axes[0], delta, full.extent, scene=scene, cmap="RdBu", vmin=-40, vmax=40,
         title="Cost of real geometry [dB]", cbar_label="RSRP change [dB]",
         label_towers=False)
axes[1].hist(delta.ravel(), bins=60, color=SUNGLOW, alpha=0.85)
axes[1].axvline(0, color=MARBLE_WHITE, lw=1)
axes[1].set_xlabel("RSRP change vs. flat, empty scene [dB]")
axes[1].set_ylabel("pixels")
axes[1].set_title(f"median {np.median(delta):.1f} dB, worst {delta.min():.0f} dB")
axes[1].grid(alpha=0.2)
fig.tight_layout()

Two things to take from that table.

**Terrain alone slightly *helps* the median.** The towers sit on high ground, so modelling
the terrain lifts the antennas — and the same terrain that raises them shadows the far
side. A flat scene is not a conservative approximation; it is a different scene.

**The damage lands on the tail, not the average.** The median moves by around a dB while
the worst pixels lose tens of dB, and the difference map shows why: the loss is
concentrated in a few terrain shadows behind the ridge, not spread evenly. That spatial
structure is the entire reason to build a twin rather than apply a correction factor to a
propagation formula — **an average is not a plan.**

## Add a third tower

The tempting fix for a weak cell edge is another site. Let us test that.

In [ ]:
new_site = Tower(name="New Site", x=250.0, y=350.0, h=25.0,
                 ground_z=float(scene.terrain_z(250.0, 350.0)), in_scene=True)
print(f"placing '{new_site.name}' at ({new_site.x:.0f}, {new_site.y:.0f}), "
      f"{new_site.h:.0f} m on {new_site.ground_z:.1f} m ground")

two = model.coverage(scene, cell_m=10.0)
three = model.coverage(scene, towers=scene.towers_in_scene() + [new_site], cell_m=10.0)

for label, c in [("2 towers", two), ("3 towers", three)]:
    print(f"\n{label}")
    print("  median RSRP     :", f"{np.median(c.best_rsrp_dbm):.1f} dBm")
    print("  5th pct RSRP    :", f"{np.percentile(c.best_rsrp_dbm, 5):.1f} dBm")
    print("  median SINR     :", f"{np.median(c.sinr_db):.1f} dB")
    print("  area SINR >= 0  :", f"{100 * c.fraction_sinr_above(0):.1f}%")
    print("  area SINR >= 10 :", f"{100 * c.fraction_sinr_above(10):.1f}%")

In [ ]:
scene3 = load_scene()
scene3.towers = scene3.towers_in_scene() + [new_site]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
plot_coverage(axes[0][0], two, scene=scene, title="RSRP — 2 towers")
plot_coverage(axes[0][1], three, scene=scene3, title="RSRP — 3 towers")
plot_sinr(axes[1][0], two, scene=scene, title="SINR — 2 towers")
plot_sinr(axes[1][1], three, scene=scene3, title="SINR — 3 towers")
fig.tight_layout()

Coverage improves; **SINR does not improve nearly as much**, and new red seams appear
where the third site fights the other two. This is the interference-limited regime the
pilot study's four-tower SINR map lands in
(`docs/renders/sinr_map.png`): once you are here, the next site buys you less than
downtilt, power tuning and a reuse plan would.

In [ ]:
from IPython.display import Image, display

p = EXAMPLES.parent / "docs" / "renders" / "sinr_map.png"
if p.exists():
    print("ray-traced, 4 towers, 2 W, 100 MHz — the same lesson with full multipath:")
    display(Image(filename=str(p), width=640))

## Where to take this

- **Move the site.** `new_site` is three numbers — sweep x/y/height and plot median SINR
  to find where a third site actually pays.
- **Change the model.** `PropagationModel` exposes `tx_power_dbm`, `bandwidth_hz`,
  `noise_figure_db`, `rx_height_m`, `eps_r`/`sigma` (ground material), and the
  `ground_reflection` mode.
- **Go interactive.** `examples/apps/coverage_explorer` is this notebook with sliders.
- **Then ray-trace it.** When a candidate plan survives here, `ulap-scope coverage` runs
  the real thing.

> ⚠️ Reminder: analytical model. No multi-bounce, no material-specific reflection or
> transmission, no scattering, no delay spread. See the caveats in the
> [study](https://app.notion.com/p/amini-updates/Digital-Twin-Propagation-Study-for-6G-Deployment-Barbados-Performing-Arts-Centre-3a5fd2e0589a8039b229e564397f3c93)
> before treating any number here — or in the ray-traced renders — as bankable.